# DST-EDL: Complete Paper Experiment & Analysis Suite

This notebook runs the complete set of experiments and analytical evaluations required to validate the paper manuscript, address reviewer objections, and construct all tables and statistics.

## 📌 Included Components:
1. **EFL-Matched Baselines (Factorial Table 2):** Trains `rigl_style_24_efl`, `static_24_edl_efl`, and `dense_edl_efl` to isolate topology vs. focal modulation.
2. **Alternative Regrowers:** Trains `guds_random_regrower` and `guds_task_gradient_regrower` to prove the necessity of KL-uniform regrowth.
3. **Out-of-Distribution (OOD):** Audits unadapted OOD on PAD-UFES-20 held-out `imgs_part_3`.
4. **Fairness Domain Adaptation:** Runs patient-grouped nested cross-validation on PAD-UFES-20 (`partition=all`) under frozen features and Layer-4 KD.
5. **Main & Softmax Baselines:** CE, Focal, Balanced Softmax, LDAM-DRW, cRT, MiSLAS, Fisher, Flexible, and R-EDL.
6. **CIFAR-100-LT 1:100:** Stress test baseline comparison.
7. **Post-Training Diagnostics:** Paired patient-level test bootstrap, mask turnover stats, and $u_a$/$u_e$ uncertainty diagnostics.

In [ ]:
# Cell 0 -- Setup & Version Locking
import os, sys, json, shutil, subprocess, shlex, time, gc
from pathlib import Path

# Detect GPU compute capability
try:
    smi = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        text=True, stderr=subprocess.DEVNULL,
    ).strip()
    major = int(smi.split('.')[0])
except Exception:
    major = 99

if major < 7:
    print(f'[WARN] GPU compute capability {smi} < 7.0; installing compatible PyTorch...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.2.2+cu121', 'torchvision==0.17.2+cu121', '--index-url', 'https://download.pytorch.org/whl/cu121'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2'])
    print('[OK] Downgraded PyTorch for compatibility.')

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:128,expandable_segments:True')
os.environ.setdefault('MALLOC_TRIM_THRESHOLD_', '0')

import torch
print(f'[OK] PyTorch {torch.__version__}, CUDA {torch.version.cuda}, GPU available={torch.cuda.is_available()}')

REPO_URL = 'https://github.com/minhduc110207/MDEP-Microglial-Driven-Evidential-Pruning.git'
REPO_ROOT = Path('/kaggle/working/MDEP-Microglial-Driven-Evidential-Pruning')

def ram_status():
    try:
        info = {}
        for line in Path('/proc/meminfo').read_text().splitlines():
            key, value = line.split(':', 1)
            info[key] = float(value.strip().split()[0]) / (1024 * 1024)
        return f"RAM available={info.get('MemAvailable', 0):.2f}GB / total={info.get('MemTotal', 0):.2f}GB"
    except Exception: return 'RAM status unavailable'

def cleanup_after_job(label=''):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[CLEANUP] {label} {ram_status()}", flush=True)

def run(cmd, env=None):
    cmd = [str(x) for x in cmd]
    print('\n$', shlex.join(cmd), flush=True)
    log_base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    log_dir = log_base / 'paper_experiment_outputs' / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    tag = Path(cmd[1]).stem if len(cmd) > 1 and cmd[1].endswith('.py') else Path(cmd[0]).stem
    log_path = log_dir / f"{time.strftime('%Y%m%d_%H%M%S')}_{tag}.log"
    merged = os.environ.copy()
    if env: merged.update({str(k): str(v) for k, v in env.items()})
    proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT), env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    important = ('[RUN]', '[INFO]', '[WARN]', '[ERROR]', '[DONE]', '[GATE', '[SKIP]', '[CHECKPOINT]', 'Traceback', 'pAUC', 'AUROC', 'AP=')
    tail = []
    line_count = 0
    with log_path.open('w', encoding='utf-8') as fh:
        for line in proc.stdout:
            line_count += 1
            fh.write(line)
            tail.append(line)
            tail = tail[-80:]
            if line_count <= 25 or any(t in line for t in important) or line_count % 200 == 0:
                print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(''.join(tail), flush=True)
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    cleanup_after_job('after sub')

if not (REPO_ROOT / '.git').exists():
    run(['git', 'clone', '--branch', 'main', '--single-branch', REPO_URL, REPO_ROOT], cwd='/kaggle/working')
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_ROOT)

os.chdir(REPO_ROOT)
KAGGLE_P100 = os.environ.get('KAGGLE_P100_MEMORY_SAFE', '1') != '0'
base_env = {'PYTHONHASHSEED': '0', 'PYTORCH_CUDA_ALLOC_CONF': 'max_split_size_mb:128,expandable_segments:True', 'MALLOC_TRIM_THRESHOLD_': '0'}
if KAGGLE_P100:
    base_env.update({'MDEP_DETERMINISTIC': '0', 'MDEP_CUDNN_BENCHMARK': '1', 'MDEP_NUM_WORKERS': '0', 'MDEP_PREFETCH_FACTOR': '2', 'OMP_NUM_THREADS': '2'})
else:
    base_env.update({'MDEP_DETERMINISTIC': '1', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'})
os.environ.update(base_env)
print(f"[MODE] KAGGLE_P100={KAGGLE_P100}; {ram_status()}")

In [ ]:
# Cell 1 -- Dataset Autodetection
INPUT_ROOT = Path('/kaggle/input')

def locate_isic(root):
    for csv in root.rglob('train-metadata.csv'):
        p = csv.parent
        if (p / 'train-image.hdf5').exists() or (p / 'train-image' / 'image').exists():
            return p
    return None

def locate_cifar(root):
    for p in root.rglob('*'):
        if p.is_dir() and all((p / x).exists() for x in ('train', 'test', 'meta')):
            return p
    return None

def locate_pad(root):
    for csv in root.rglob('metadata.csv'):
        p = csv.parent
        if any(x in str(p).lower() for x in ('isic', 'cifar')):
            continue
        if all((p / x).exists() for x in ('imgs_part_1', 'imgs_part_2', 'imgs_part_3')):
            return p
    return None

ISIC_ROOT = locate_isic(INPUT_ROOT)
CIFAR_FOLDER = locate_cifar(INPUT_ROOT)
PAD_ROOT = locate_pad(INPUT_ROOT)

assert ISIC_ROOT, "ISIC 2024 dataset not found under /kaggle/input."
os.environ['ISIC_ROOT'] = str(ISIC_ROOT)

if CIFAR_FOLDER:
    cifar_root = Path('/kaggle/working/cifar_data')
    cifar_root.mkdir(parents=True, exist_ok=True)
    link = cifar_root / 'cifar-100-python'
    if not link.exists():
        link.symlink_to(CIFAR_FOLDER, target_is_directory=True)
    os.environ['CIFAR_ROOT'] = str(cifar_root)
else:
    print("[WARN] CIFAR-100 not found; CIFAR generalization suite will be skipped.")

PAD_META = PAD_ROOT / 'metadata.csv' if PAD_ROOT else None
print('[OK] ISIC_ROOT =', ISIC_ROOT)
print('[OK] CIFAR_ROOT =', os.environ.get('CIFAR_ROOT'))
print('[OK] PAD_ROOT =', PAD_ROOT)

In [ ]:
# Cell 2 -- Execution controls and resume checks
SEEDS = [42, 123, 456]
SPLIT_SEED = 42
ISIC_EPOCHS = 40
ISIC_PROTOCOL_PROFILE = 'nvidia_v3' # Required for OOD, TRT, and exact 2:4 structured sparsity
ISIC_BATCH_SIZE = 32
CIFAR_BATCH_SIZE = 128

RUN_MAIN_TABLE = True
RUN_REQUIRED_ABLATIONS = True
RUN_SOFTMAX_APPENDIX = True
RUN_CIFAR100_LT = CIFAR_FOLDER is not None
RUN_OOD = PAD_ROOT is not None
RUN_PAD_ADAPTATION = PAD_ROOT is not None
RUN_LAYER4_KD = PAD_ROOT is not None

OUT = Path('/kaggle/working/paper_experiment_outputs')
ISIC_OUT = OUT / 'isic'
FAIR_SUFFIX = '_fair_v3_nvidia24'
CIFAR_SUFFIX = '_fair_v3_nvidia24_2026_07_09'
PROTOCOL = 'isic_fair_v3_nvidia24_2026_07_09'

def isic_seed_valid(name, seed, require_model=False):
    d = ISIC_OUT / f"{name}{FAIR_SUFFIX}" / f"seed_{seed}"
    p = d / 'metrics.json'
    if not p.exists() or (require_model and not (d / 'model_state.pth').exists()):
        return False
    try:
        x = json.loads(p.read_text())
        return (x.get('protocol_version') == PROTOCOL and x.get('epochs') == ISIC_EPOCHS and
                x.get('split_seed') == SPLIT_SEED and x.get('batch_size') == ISIC_BATCH_SIZE)
    except Exception: return False

def run_isic(names, save_model=False):
    for seed in SEEDS:
        pending = [n for n in names if not isic_seed_valid(n, seed, require_model=save_model)]
        if not pending: continue
        cmd = [sys.executable, '-u', 'experiments/isic_paper_experiments.py']
        for name in pending:
            cmd += ['--experiment', name]
        cmd += ['--epochs', str(ISIC_EPOCHS), '--batch_size', str(ISIC_BATCH_SIZE), '--lr', '4e-5',
                '--seed', str(seed), '--split_seed', str(SPLIT_SEED), '--subsample_scope', 'train',
                '--subsample_ratio', '20', '--structural_proxy_batches', '4', '--checkpoint_selection', 'last',
                '--protocol_profile', ISIC_PROTOCOL_PROFILE, '--run_suffix', FAIR_SUFFIX]
        if not save_model: cmd.append('--no_save_model')
        run(cmd)

print('Execution configurations initialized.')

In [ ]:
# Cell 3 -- Run Factorial Main Table (Baselines + EFL-matched ones)
MAIN_RUNS = [
    "dense_edl", "static_24_edl", "rigl_style_24", "full_guds",
    "dense_edl_efl", "static_24_edl_efl", "rigl_style_24_efl"
]

if RUN_MAIN_TABLE:
    # Save checkpoints for projection and OOD tasks
    run_isic(MAIN_RUNS, save_model=True)
    print('[DONE] Main and EFL-matched baselines complete.')
else:
    print('[SKIP] Main Table runs.')

In [ ]:
# Cell 4 -- OOD on held-out PAD-UFES imgs_part_3
def ood_complete(seed):
    candidates = list((OUT / 'external_validation').glob(f'external_validation_seed_{seed}_imgs_part_3_*.json'))
    return len(candidates) > 0

if RUN_OOD:
    for seed in SEEDS:
        if ood_complete(seed):
            print(f'[SKIP] OOD seed={seed}')
            continue
        checkpoint = ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / f"seed_{seed}" / "model_state.pth"
        run([sys.executable, '-u', 'experiments/run_external_validation.py',
             '--model_path', checkpoint, '--seed', str(seed), '--split_seed', str(SPLIT_SEED),
             '--custom_image_folder', PAD_ROOT, '--pad_ufes_csv', PAD_META,
             '--pad_ufes_partition', 'imgs_part_3', '--knn_primary_layer', 'layer3',
             '--primary_ood_score', 'knn_layer3'])
    print('[DONE] OOD held-out evaluation complete.')
else:
    print('[SKIP] OOD evaluation.')

In [ ]:
# Cell 5 -- Frozen-feature Domain Adaptation on PAD-UFES partition=all
ADAPT_SUMMARY = OUT / 'pad_adaptation/pad_adaptation_summary.json'

if RUN_PAD_ADAPTATION:
    if ADAPT_SUMMARY.exists():
        print(f'[SKIP] PAD adaptation summary found.')
    else:
        ckpt_tmpl = str(ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / "seed_{seed}" / "model_state.pth")
        run([sys.executable, '-u', 'experiments/run_pad_adaptation.py',
             '--pad_root', PAD_ROOT, '--pad_csv', PAD_META, '--partition', 'all',
             '--model_path', ckpt_tmpl, '--seeds', *map(str, SEEDS), '--split_seed', str(SPLIT_SEED),
             '--target_mode', 'diagnosis6', '--feature_layer', 'auto', '--head', 'linear',
             '--outer_folds', '5', '--inner_folds', '3', '--fairness_min_group_size', '20',
             '--fairness_min_class_size', '10', '--target_sensitivity', '0.80',
             '--bootstrap_repeats', '200', '--batch_size', '32', '--num_workers', '0',
             '--train_domain_head'])
    print('[DONE] PAD frozen-feature Domain Adaptation complete.')
else:
    print('[SKIP] PAD Domain Adaptation.')

In [ ]:
# Cell 6 -- Layer-4 Knowledge Distillation Adaptation on PAD-UFES partition=all
KD_SUMMARY = OUT / 'pad_layer4_kd/pad_layer4_kd_summary.json'

if RUN_LAYER4_KD:
    if KD_SUMMARY.exists():
        print(f'[SKIP] PAD Layer-4 KD summary found.')
    else:
        ckpt_tmpl = str(ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / "seed_{seed}" / "model_state.pth")
        run([sys.executable, '-u', 'experiments/run_pad_layer4_kd.py',
             '--pad_root', PAD_ROOT, '--pad_csv', PAD_META, '--partition', 'all',
             '--model_path', ckpt_tmpl, '--seeds', *map(str, SEEDS), '--split_seed', str(SPLIT_SEED),
             '--target_mode', 'diagnosis6', '--outer_folds', '5', '--inner_folds', '3',
             '--epochs', '12', '--lr', '1e-5', '--kd_weight', '2.0', '--kd_temperature', '2.0',
             '--target_sensitivity', '0.80', '--bootstrap_repeats', '200',
             '--batch_size', '16', '--num_workers', '0'])
    print('[DONE] PAD Layer-4 KD complete.')
else:
    print('[SKIP] PAD Layer-4 KD.')

In [ ]:
# Cell 7 -- Required & Alternate Regrowth Ablations
ABLATION_RUNS = [
    "guds_without_pruner", "guds_without_regrower", "guds_asymmetric_kl",
    "guds_without_efl", "guds_without_anticryst",
    "guds_random_regrower", "guds_task_gradient_regrower"
]

if RUN_REQUIRED_ABLATIONS:
    run_isic(ABLATION_RUNS, save_model=False)
    print('[DONE] Required and regrowth ablations complete.')
else:
    print('[SKIP] Required Ablations.')

In [ ]:
# Cell 8 -- Softmax Long-Tailed baselines
SOFTMAX_RUNS = [
    "standard_ce", "focal_loss", "class_balanced_ce", "balanced_softmax",
    "ldam_drw", "decoupled_crt", "mislas"
]

if RUN_SOFTMAX_APPENDIX:
    run_isic(SOFTMAX_RUNS, save_model=False)
    print('[DONE] Softmax long-tail baselines complete.')
else:
    print('[SKIP] Softmax baselines.')

In [ ]:
# Cell 9 -- CIFAR-100-LT 1:100 Generalization test
CIFAR_MODELS = ["standard_ce", "dense_edl", "full_guds"]
CIFAR_OUT = OUT / 'cifar' / 'ir100'

def cifar_seed_valid(name, seed):
    p = CIFAR_OUT / f"{name}{CIFAR_SUFFIX}" / f"seed_{seed}" / 'metrics.json'
    if not p.exists(): return False
    try:
        x = json.loads(p.read_text())
        return x.get('epochs') == 100 and x.get('split_seed') == SPLIT_SEED
    except Exception: return False

if RUN_CIFAR100_LT:
    for seed in SEEDS:
        missing_cifar = [n for n in CIFAR_MODELS if not cifar_seed_valid(n, seed)]
        if not missing_cifar: continue
        cmd = [sys.executable, '-u', 'experiments/generalization_paper_suite.py', '--benchmark', 'cifar']
        for name in missing_cifar: cmd += ['--experiment', name]
        cmd += ['--ratio', '100', '--epochs', '100', '--batch_size', '128',
                '--seed', str(seed), '--split_seed', str(SPLIT_SEED), '--run_suffix', CIFAR_SUFFIX]
        run(cmd, env={'CIFAR_ROOT': os.environ['CIFAR_ROOT']})
    print('[DONE] CIFAR-100-LT suite complete.')
else:
    print('[SKIP] CIFAR-100-LT.')

In [ ]:
# Cell 10 -- Post-Training Analytical Evaluation & Bootstrap CIs
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from guds_edl_core import compute_isic_pauc

def run_paired_bootstrap(pred_path_a, pred_path_b, n_bootstraps=200, seed=42):
    """Perform patient-level bootstrap CI to establish statistical reliability."""
    if not pred_path_a.exists() or not pred_path_b.exists():
        return None
    df_a = pd.read_csv(pred_path_a)
    df_b = pd.read_csv(pred_path_b)
    
    # Align patients and indices
    patients = df_a['patient_id'].unique()
    np.random.seed(seed)
    
    a_groups = {pid: grp for pid, grp in df_a.groupby('patient_id')}
    b_groups = {pid: grp for pid, grp in df_b.groupby('patient_id')}
    
    diffs_pauc = []
    diffs_auroc = []
    for _ in range(n_bootstraps):
        boot_patients = np.random.choice(patients, size=len(patients), replace=True)
        # Concatenate selected patient groups
        boot_a = pd.concat([a_groups[pid] for pid in boot_patients if pid in a_groups], ignore_index=True)
        boot_b = pd.concat([b_groups[pid] for pid in boot_patients if pid in b_groups], ignore_index=True)
        
        pauc_a = compute_isic_pauc(boot_a['y_true'].to_numpy(), boot_a['prob_1'].to_numpy())
        pauc_b = compute_isic_pauc(boot_b['y_true'].to_numpy(), boot_b['prob_1'].to_numpy())
        diffs_pauc.append(pauc_a - pauc_b)
        
        auroc_a = roc_auc_score(boot_a['y_true'].to_numpy(), boot_a['prob_1'].to_numpy())
        auroc_b = roc_auc_score(boot_b['y_true'].to_numpy(), boot_b['prob_1'].to_numpy())
        diffs_auroc.append(auroc_a - auroc_b)
        
    ci_pauc = np.percentile(diffs_pauc, [2.5, 97.5])
    ci_auroc = np.percentile(diffs_auroc, [2.5, 97.5])
    return {
        'mean_pauc_diff': np.mean(diffs_pauc),
        'ci_pauc_95': ci_pauc.tolist(),
        'mean_auroc_diff': np.mean(diffs_auroc),
        'ci_auroc_95': ci_auroc.tolist(),
        'pauc_significant': bool(ci_pauc[0] > 0)
    }

def compute_mask_similarities(ckpt_a, ckpt_b):
    """Compute Hamming distance and Jaccard similarity between two checkpoints to analyze mask turnover."""
    if not ckpt_a.exists() or not ckpt_b.exists():
        return None
    sa = torch.load(ckpt_a, map_location='cpu') if isinstance(ckpt_a, (str, Path)) else ckpt_a
    sb = torch.load(ckpt_b, map_location='cpu') if isinstance(ckpt_b, (str, Path)) else ckpt_b
    hamming = []
    jaccard = []
    for k in sa.keys():
        if k.endswith('.mask') and k in sb:
            ma = sa[k].bool()
            mb = sb[k].bool()
            intersection = (ma & mb).sum().item()
            union = (ma | mb).sum().item()
            jaccard.append(intersection / max(union, 1))
            hamming.append((ma != mb).sum().item())
    return {
        'mean_hamming': float(np.mean(hamming)) if hamming else 0.0,
        'mean_jaccard': float(np.mean(jaccard)) if jaccard else 0.0
    }

def compute_uncertainty_diagnostics(pred_path):
    """Analyze epistemic and aleatoric uncertainties on correct vs incorrect predictions."""
    if not pred_path.exists():
        return None
    df = pd.read_csv(pred_path)
    if 'u_a' not in df.columns or 'u_e' not in df.columns:
        return None
    df['correct'] = (df['y_pred_balanced'] == df['y_true']).astype(int)
    correct = df[df['correct'] == 1]
    incorrect = df[df['correct'] == 0]
    c0 = df[df['y_true'] == 0]
    c1 = df[df['y_true'] == 1]
    return {
        'correct_mean_ua': float(correct['u_a'].mean()),
        'incorrect_mean_ua': float(incorrect['u_a'].mean()),
        'correct_mean_ue': float(correct['u_e'].mean()),
        'incorrect_mean_ue': float(incorrect['u_e'].mean()),
        'class_0_mean_ua': float(c0['u_a'].mean()),
        'class_1_mean_ua': float(c1['u_a'].mean()),
        'class_0_mean_ue': float(c0['u_e'].mean()),
        'class_1_mean_ue': float(c1['u_e'].mean())
    }

print('Analytical diagnostics helper functions loaded.')

In [ ]:
# Cell 11 -- Compute and Save all Analytical Reports
reports = {}

# 1. Bootstrap Comparison: DST-EDL (full_guds) vs RigL-style 2:4 (without EFL)
bootstrap_results = []
for seed in SEEDS:
    path_guds = ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / f"seed_{seed}" / "test_predictions.csv"
    path_rigl = ISIC_OUT / f"rigl_style_24{FAIR_SUFFIX}" / f"seed_{seed}" / "test_predictions.csv"
    res = run_paired_bootstrap(path_guds, path_rigl, seed=seed)
    if res: bootstrap_results.append(res)
reports['bootstrap_guds_vs_rigl'] = bootstrap_results

# 2. Mask Turnover Analysis: Cross-seed Jaccard of full_guds (42 vs 123 vs 456)
sa = ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / "seed_42" / "model_state.pth"
sb = ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / "seed_123" / "model_state.pth"
sc = ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / "seed_456" / "model_state.pth"
reports['mask_turnover_cross_seed'] = {
    'seed_42_vs_123': compute_mask_similarities(sa, sb),
    'seed_42_vs_456': compute_mask_similarities(sa, sc)
}

# 3. Evidential Uncertainty Diagnostics
ue_res = []
for seed in SEEDS:
    p = ISIC_OUT / f"full_guds{FAIR_SUFFIX}" / f"seed_{seed}" / "test_predictions.csv"
    res = compute_uncertainty_diagnostics(p)
    if res: ue_res.append(res)
reports['uncertainty_diagnostics'] = ue_res

# Write reports
(OUT / "analytical_diagnostics_report.json").write_text(json.dumps(reports, indent=2))
print('[DONE] Saved analytical report to', OUT / "analytical_diagnostics_report.json")

In [ ]:
# Cell 12 -- Display Summary Tables (Ready to Copy-Paste!)
import pandas as pd
import json
from pathlib import Path

ISIC_OUT = Path('/kaggle/working/paper_experiment_outputs/isic')
FAIR_SUFFIX = '_fair_v3_nvidia24'
SEEDS = [42, 123, 456]

def get_metrics_for_experiment(name):
    rows = []
    for seed in SEEDS:
        p = ISIC_OUT / f"{name}{FAIR_SUFFIX}" / f"seed_{seed}" / 'metrics.json'
        if p.exists():
            x = json.loads(p.read_text())
            m = x.get('metrics', {})
            rows.append({
                'pAUC': m.get('pauc'),
                'Macro AUROC': m.get('macro_auroc'),
                'PR AUC': m.get('pr_auc'),
                'ECE': m.get('ece_adaptive'),
                'Bal. Acc': m.get('balanced_accuracy_default', m.get('balanced_accuracy'))
            })
    if not rows: return None
    df = pd.DataFrame(rows)
    res = {}
    for col in df.columns:
        res[col] = f"{df[col].mean():.4f} \u00b1 {df[col].std():.4f}"
    return res

print("================================================================================")
print("TABLE 2: FACTORIAL TABLE (Topology under Matched and Focal Objectives)")
print("================================================================================")
t2_rows = []
for name, label in [
    ('dense_edl', 'Dense EDL (base loss)'),
    ('static_24_edl', 'Static 2:4 (base loss)'),
    ('rigl_style_24', 'RigL-style 2:4 (base loss)'),
    ('guds_without_efl', 'DST-EDL w/o EFL (base loss)'),
    ('dense_edl_efl', 'Dense EDL + EFL'),
    ('static_24_edl_efl', 'Static 2:4 + EFL'),
    ('rigl_style_24_efl', 'RigL-style 2:4 + EFL'),
    ('full_guds', 'DST-EDL (full)')
]:
    metrics = get_metrics_for_experiment(name)
    if metrics:
        metrics['Model'] = label
        t2_rows.append(metrics)
if t2_rows:
    display(pd.DataFrame(t2_rows)[['Model', 'Bal. Acc', 'pAUC', 'Macro AUROC', 'ECE']])

print("\n================================================================================")
print("REGROWER COMPARISON ABLATION (Proof of KL-uniform necessity)")
print("================================================================================")
ab_rows = []
for name, label in [
    ('full_guds', 'KL-Uniform Regrowth (Proposed)'),
    ('guds_random_regrower', 'Random Regrowth (Ablation)'),
    ('guds_task_gradient_regrower', 'Task-Gradient Regrowth (RigL alternative)'),
    ('guds_class_conditioned_regrower', 'Class-Conditioned Regrowth')
]:
    metrics = get_metrics_for_experiment(name)
    if metrics:
        metrics['Regrower'] = label
        ab_rows.append(metrics)
if ab_rows:
    display(pd.DataFrame(ab_rows)[['Regrower', 'Bal. Acc', 'pAUC', 'Macro AUROC', 'ECE']])

print("\n================================================================================")
print("POST-TRAINING ANALYTICAL REPORT SUMMARY")
print("================================================================================")
rep_path = Path('/kaggle/working/paper_experiment_outputs/analytical_diagnostics_report.json')
if rep_path.exists():
    rep = json.loads(rep_path.read_text())
    print("\n1. Patient-Level Bootstrap (Guds vs RigL):")
    for i, seed in enumerate([42, 123, 456]):
        if i < len(rep['bootstrap_guds_vs_rigl']):
            r = rep['bootstrap_guds_vs_rigl'][i]
            print(f"  Seed {seed}: pAUC mean diff = {r['mean_pauc_diff']:.4f}, 95% CI = {r['ci_pauc_95']}, significant={r['pauc_significant']}")
            
    print("\n2. Mask Jaccard similarity across seeds:")
    print(f"  Seed 42 vs 123 Jaccard Similarity: {rep['mask_turnover_cross_seed']['seed_42_vs_123']['mean_jaccard']:.4f}")
    print(f"  Seed 42 vs 456 Jaccard Similarity: {rep['mask_turnover_cross_seed']['seed_42_vs_456']['mean_jaccard']:.4f}")
    
    print("\n3. Epistemic (u_e) and Aleatoric (u_a) uncertainty by predictions correctness:")
    for i, seed in enumerate([42, 123, 456]):
        if i < len(rep['uncertainty_diagnostics']):
            r = rep['uncertainty_diagnostics'][i]
            print(f"  Seed {seed}: Correct cases mean u_a = {r['correct_mean_ua']:.4f}, Incorrect cases mean u_a = {r['incorrect_mean_ua']:.4f}")
